In [19]:
# ==========================================================
# PRODUCTION-GRADE MENSTRUAL INTELLIGENCE ENGINE
# Multi-Task LSTM with Edge-Case Handling
# ==========================================================

# ==============================
# 1️⃣ IMPORT LIBRARIES
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Input,
    Masking, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping
from datetime import datetime, timedelta

np.random.seed(42)
tf.random.set_seed(42)

# ==========================================================
# 2️⃣ LOAD DATA
# ==========================================================

df = pd.read_csv("FedCycleData071012.csv", encoding="utf-8-sig")

print("Original Dataset Shape:", df.shape)

# ==========================================================
# 3️⃣ SELECT ONLY APP-COLLECTABLE FEATURES
# ==========================================================

usable_columns = [
    "ClientID",
    "CycleNumber",
    "LengthofCycle",
    "EstimatedDayofOvulation",
    "LengthofLutealPhase",
    "LengthofMenses",
    "Age",
    "BMI",
    "CycleWithPeakorNot"
]

df = df[usable_columns]

# Convert to numeric safely
numeric_cols = [
    "LengthofCycle",
    "EstimatedDayofOvulation",
    "LengthofLutealPhase",
    "LengthofMenses",
    "Age",
    "BMI"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ==========================================================
# 4️⃣ BIOLOGICAL FILTERING (REALISTIC CONSTRAINTS)
# ==========================================================

df = df[
    (df["LengthofCycle"].between(18, 60)) &
    (df["LengthofMenses"].between(2, 15))
]

# Fill remaining missing numeric values with median
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Sort properly
df = df.sort_values(["ClientID", "CycleNumber"]).reset_index(drop=True)

print("Cleaned Dataset Shape:", df.shape)

# ==========================================================
# 5️⃣ ANALYZE USER HISTORY DISTRIBUTION
# ==========================================================

cycle_counts = df.groupby("ClientID").size()

print("\nCycle Count Statistics:")
print(cycle_counts.describe())

# ==========================================================
# 6️⃣ BUILD SEQUENCE DATASET
# ==========================================================

WINDOW_SIZE = 3  # smaller window to preserve data

X = []
y = []

for client in df["ClientID"].unique():

    client_data = df[df["ClientID"] == client].reset_index(drop=True)

    if len(client_data) > WINDOW_SIZE:

        for i in range(len(client_data) - WINDOW_SIZE):

            window = client_data.iloc[i:i+WINDOW_SIZE]
            target = client_data.iloc[i+WINDOW_SIZE]

            sequence = window[[
                "LengthofCycle",
                "LengthofMenses",
                "EstimatedDayofOvulation",
                "LengthofLutealPhase",
                "Age",
                "BMI",
                "CycleWithPeakorNot"
            ]].values

            target_values = [
                target["LengthofCycle"],
                target["EstimatedDayofOvulation"],
                target["LengthofLutealPhase"],
                target["LengthofMenses"]
            ]

            X.append(sequence)
            y.append(target_values)

X = np.array(X)
y = np.array(y)

print("\nFinal Sequence Shape:", X.shape)

if len(X) == 0:
    raise ValueError("No valid sequences created. Reduce WINDOW_SIZE or inspect dataset.")

# ==========================================================
# 7️⃣ ROBUST SCALING (handles outliers better)
# ==========================================================

feature_scaler = RobustScaler()
target_scaler = RobustScaler()

X_reshaped = X.reshape(-1, X.shape[2])
X_scaled = feature_scaler.fit_transform(X_reshaped)
X_scaled = X_scaled.reshape(X.shape)

y_scaled = target_scaler.fit_transform(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

# ==========================================================
# 8️⃣ MODEL ARCHITECTURE (Bidirectional LSTM)
# ==========================================================

input_layer = Input(shape=(WINDOW_SIZE, X.shape[2]))

x = Masking(mask_value=0.0)(input_layer)

x = Bidirectional(LSTM(128, return_sequences=True))(x)
x = Dropout(0.3)(x)

x = Bidirectional(LSTM(64))(x)
x = Dropout(0.3)(x)

x = Dense(64, activation="relu")(x)

output = Dense(4)(x)  # Multi-output regression

model = Model(inputs=input_layer, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

# ==========================================================
# 9️⃣ TRAINING
# ==========================================================

early_stop = EarlyStopping(patience=8, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=60,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ==========================================================
# 🔟 EVALUATION
# ==========================================================

pred_scaled = model.predict(X_test)
pred = target_scaler.inverse_transform(pred_scaled)
true = target_scaler.inverse_transform(y_test)

print("\nEvaluation (MAE per Target):")

labels = ["Cycle", "Ovulation", "Luteal", "Menses"]

for i, label in enumerate(labels):
    mae = mean_absolute_error(true[:, i], pred[:, i])
    print(f"{label} MAE: {mae:.2f} days")

# ==========================================================
# 1️⃣1️⃣ LOSS VISUALIZATION
# ==========================================================

plt.figure()
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Training vs Validation Loss")
plt.legend(["Train", "Validation"])
plt.show()

# ==========================================================
# 1️⃣2️⃣ APP PREDICTION FUNCTION
# ==========================================================

def predict_user_next_cycle_with_dates(user_history, last_period_start_date):
    """
    user_history: shape (WINDOW_SIZE, 7)
    last_period_start_date: string format 'YYYY-MM-DD'
    """

    user_history = np.array(user_history)

    if user_history.shape != (WINDOW_SIZE, X.shape[2]):
        raise ValueError(f"Input must be shape ({WINDOW_SIZE}, {X.shape[2]})")

    # Scale input
    user_scaled = feature_scaler.transform(
        user_history.reshape(-1, X.shape[2])
    ).reshape(1, WINDOW_SIZE, X.shape[2])

    # Predict
    pred_scaled = model.predict(user_scaled)
    pred = target_scaler.inverse_transform(pred_scaled)

    predicted_cycle = round(float(pred[0][0]))
    predicted_ovulation_day = round(float(pred[0][1]))
    predicted_luteal = round(float(pred[0][2]))
    predicted_menses = round(float(pred[0][3]))

    # Convert last period date
    last_period_date = datetime.strptime(last_period_start_date, "%Y-%m-%d")

    # Calculate predicted next period start
    next_period_date = last_period_date + timedelta(days=predicted_cycle)

    # Ovulation date
    ovulation_date = last_period_date + timedelta(days=predicted_ovulation_day)

    # Fertility window (approx 5 days before ovulation)
    fertile_start = ovulation_date - timedelta(days=5)
    fertile_end = ovulation_date + timedelta(days=1)

    return {
        "Predicted Cycle Length (days)": predicted_cycle,
        "Next Period Start Date": next_period_date.strftime("%Y-%m-%d"),
        "Predicted Ovulation Date": ovulation_date.strftime("%Y-%m-%d"),
        "Fertile Window Start": fertile_start.strftime("%Y-%m-%d"),
        "Fertile Window End": fertile_end.strftime("%Y-%m-%d"),
        "Predicted Menses Duration (days)": predicted_menses,
        "Predicted Luteal Phase (days)": predicted_luteal
    }

# ==========================================================
# 1️⃣3️⃣ SAMPLE TEST
# ==========================================================

sample_input = X[0]
print("\nSample Prediction:")
# Fix: Use the correct function name and provide a sample date for last_period_start_date
print(predict_user_next_cycle_with_dates(sample_input, "2023-01-01"))

Original Dataset Shape: (1665, 80)
Cleaned Dataset Shape: (1661, 9)

Cycle Count Statistics:
count    159.000000
mean      10.446541
std        7.034857
min        1.000000
25%        5.000000
50%       12.000000
75%       13.000000
max       45.000000
dtype: float64

Final Sequence Shape: (1219, 3, 7)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 3, 7)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_2         │ (None, 3, 7)      │          0 │ input_layer_2[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking_2 (Masking) │ (None, 3, 7)      │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any_2 (Any)         │ (None, 3)         │          0 │ not_equal_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 3, 256)    │    139,264 │ masking_2[0][0],  │
│ (Bidirectional)     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 3, 256)    │          0 │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 128)       │    164,352 │ dropout_4[0][0],  │
│ (Bidirectional)     │                   │            │ any_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 128)       │          0 │ bidirectional_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      8,256 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 4)         │        260 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 312,132 (1.19 MB)

 Trainable params: 312,132 (1.19 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 6s 47ms/step - loss: 0.8357 - mae: 0.5999 - val_loss: 0.7907 - val_mae: 0.6102
Epoch 2/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - loss: 0.6534 - mae: 0.5482 - val_loss: 0.7458 - val_mae: 0.5867
Epoch 3/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.6265 - mae: 0.5259 - val_loss: 0.7397 - val_mae: 0.5788
Epoch 4/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.6125 - mae: 0.5201 - val_loss: 0.7312 - val_mae: 0.5735
Epoch 5/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.5944 - mae: 0.5155 - val_loss: 0.7320 - val_mae: 0.5760
Epoch 6/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.5866 - mae: 0.5120 - val_loss: 0.7302 - val_mae: 0.5739
Epoch 7/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.5874 - mae: 0.5151 - val_loss: 0.7321 - val_mae: 0.5761
Epoch 8/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.5944 - mae: 0.5126 - val_loss: 0.7307 - val_mae: 0.5757
Epoch 9/60
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.568

KeyboardInterrupt: 

In [ ]:
# ==============================
# SAVE MODELS (BACKEND SAFE)
# ==============================

import joblib
import os
# Save LSTM in native Keras format
model.export("cycle_model")

# Save scalers
joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")

print("All models saved successfully")

Saved artifact at 'cycle_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 3, 7), dtype=tf.float32, name='keras_tensor_10')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  135609209175312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609190152912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609209175696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609190153680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206571280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206573776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206573968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206572048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206574928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206575120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135609206574352: Tensor

In [31]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


# =====================
# LOAD DATA
# =====================

df = pd.read_csv("PCOS_data.csv")

# Clean column names by stripping whitespace
df.columns = df.columns.str.strip()


# =====================
# SELECT SAME FEATURES USED BY APP
# =====================

features = [
    "Age (yrs)",
    "BMI",
    "Weight gain(Y/N)",
    "hair growth(Y/N)",
    "Skin darkening (Y/N)",
    "Hair loss(Y/N)",
    "Pimples(Y/N)",
    "Fast food (Y/N)",
    "Reg.Exercise(Y/N)"
]

target = "PCOS (Y/N)"

df = df[features + [target]].copy()


# =====================
# CLEAN DATA
# =====================

df = df.fillna(df.median())


# =====================
# SPLIT DATA
# =====================

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# =====================
# MODEL
# =====================

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42
)

model.fit(X_train, y_train)


# =====================
# EVALUATION
# =====================

pred = model.predict(X_test)

print(classification_report(y_test, pred))


# =====================
# SAVE MODEL (SAFE)
# =====================

joblib.dump(model, "pcos_model.pkl", compress=3)

print("Model saved successfully")

              precision    recall  f1-score   support

           0       0.87      0.96      0.91        77
           1       0.88      0.66      0.75        32

    accuracy                           0.87       109
   macro avg       0.87      0.81      0.83       109
weighted avg       0.87      0.87      0.87       109

Model saved successfully


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 45 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Sl. No                  541 non-null    int64  
 1   Patient File No.        541 non-null    int64  
 2   PCOS (Y/N)              541 non-null    int64  
 3    Age (yrs)              541 non-null    int64  
 4   Weight (Kg)             541 non-null    float64
 5   Height(Cm)              541 non-null    float64
 6   BMI                     541 non-null    float64
 7   Blood Group             541 non-null    int64  
 8   Pulse rate(bpm)         541 non-null    int64  
 9   RR (breaths/min)        541 non-null    int64  
 10  Hb(g/dl)                541 non-null    float64
 11  Cycle(R/I)              541 non-null    int64  
 12  Cycle length(days)      541 non-null    int64  
 13  Marraige Status (Yrs)   540 non-null    float64
 14  Pregnant(Y/N)           541 non-null    in